In [ ]:
# Last updated: 2026-04-02 14:30 UTC

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Running on CPU.")

GPU is available!
GPU Name: NVIDIA A100-SXM4-40GB


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Clone repo and install
!git clone https://github.com/SlaviXG/thinking-inside-the-box.git
%cd thinking-inside-the-box
!pip install -e . -q

Cloning into 'thinking-inside-the-box'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 264 (delta 154), reused 190 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (264/264), 759.27 KiB | 8.08 MiB/s, done.
Resolving deltas: 100% (154/154), done.
/content/thinking-inside-the-box
  Preparing metadata (setup.py) ... done
   в”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓ 60.7/60.7 MB 43.2 MB/s eta 0:00:00
   в”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓв”Ѓ 7.6/7.6 MB 124.4 MB/s eta 0:00:00


In [1]:
from google.colab import drive
import os
from datetime import datetime

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/thinking-inside-the-box"
RUN_DATE  = datetime.now().strftime("%Y-%m-%d")
RUN_DIR   = f"{DRIVE_DIR}/{RUN_DATE}"

os.makedirs(f"{RUN_DIR}/adapters", exist_ok=True)
print(f"Drive mounted. Results -> {RUN_DIR}")


Mounted at /content/drive
Drive mounted. Checkpoints will be saved to /content/drive/MyDrive/thinking-inside-the-box


In [ ]:
# Take your kaggle.json from Drive, then:
!mkdir -p ~/.kaggle
!cp {DRIVE_DIR}/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p data
!kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml \
    -f LI-Small_Trans.csv -p data/ --unzip
!ls -lh data/

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
License(s): Community Data License Agreement - Sharing - Version 1.0
100% 620M/620M [00:08<00:00, 79.3MB/s]

total 621M
-rw-r--r-- 1 root root 621M Jul  8  2025 LI-Small_Trans.csv


In [ ]:
# Prevents Colab idle timeout during long runs.
# Run this before starting benchmarks, then leave the tab open.
from IPython.display import Javascript
display(Javascript("""function keepAlive() {
    document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(keepAlive, 60000);
"""))

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from src.config import Config
from src.graph.networkx_store import NetworkXGraphStore
from src.data.aml_ingestor import AMLIngestor

# Minimal synthetic transactions - mirrors IBM AML schema
mock_rows = [
    {"Timestamp": "2023-01-01", "From Bank": 1, "Account": "A1",
      "To Bank": 2, "Account.1": "A2", "Amount Received": 100.0,
      "Receiving Currency": "USD", "Amount Paid": 100.0,
      "Payment Currency": "USD", "Payment Format": "Wire", "Is Laundering": 0},
    {"Timestamp": "2023-01-02", "From Bank": 1, "Account": "A1",
      "To Bank": 3, "Account.1": "A3", "Amount Received": 5000.0,
      "Receiving Currency": "USD", "Amount Paid": 5000.0,
      "Payment Currency": "BTC", "Payment Format": "Crypto", "Is Laundering": 1},
    {"Timestamp": "2023-01-03", "From Bank": 2, "Account": "A2",
      "To Bank": 3, "Account.1": "A3", "Amount Received": 200.0,
      "Receiving Currency": "EUR", "Amount Paid": 200.0,
      "Payment Currency": "EUR", "Payment Format": "ACH", "Is Laundering": 0},
]
df = pd.DataFrame(mock_rows)

config = Config(graph_backend="networkx", bank_id=0)
store = NetworkXGraphStore(config)
store.connect()  # initialises the graph

ingestor = AMLIngestor(config)
nodes = ingestor.prepare_nodes(df)
edges = ingestor.prepare_edges(df)

store.create_schema()
store.ingest(nodes, edges)

context = store.retrieve_context("A1", limit=10)
print("=== RAG context for A1 ===")
print(context)

store.close()
print("\nSmoke test passed.")

=== RAG context for A1 ===
Transaction History for Account A1:
- A1 sent 100.0 USD (Wire) to A2 at 2023-01-01
- A1 sent 5000.0 BTC (Crypto) to A3 at 2023-01-02
- A2 sent 200.0 EUR (ACH) to A3 at 2023-01-03


Smoke test passed.


In [ ]:
from src.config import Config
from src.graph.factory import GraphStoreFactory
from src.model.model_loader import load_model, load_tokenizer, attach_lora
from src.data.aml_ingestor import AMLIngestor
from src.pipeline.investigation import InvestigationPipeline

config = Config(
    csv_path="data/LI-Small_Trans.csv",
    graph_backend="kuzu",
    bank_id=0,              # all banks
    retrieval_limit=20,
    max_new_tokens=1024,
)

# Load model (downloads ~5GB on first run)
# Skip if already loaded in this session
if "model" not in dir() or model is None:
    print("Loading model...")
    model = load_model(config)
    tokenizer = load_tokenizer(config)
    model = attach_lora(model, config)
    print("Model ready.")
else:
    print("Model already loaded, skipping.")

# Build graph
print("\nBuilding graph...")
ingestor = AMLIngestor(config)
df = ingestor.load_partition()
graph_store = GraphStoreFactory.create(config)
ingestor.run_from_df(graph_store, df)

# Pick one account from each class to investigate
train_df, _, test_df = ingestor.split(df)
clean_account = test_df[test_df["label"] == 0]["account_id"].iloc[0]
suspicious_account = test_df[test_df["label"] == 1]["account_id"].iloc[0]

pipeline = InvestigationPipeline(graph_store, model, tokenizer, config)

print(f"\n--- Investigating CLEAN account: {clean_account} ---")
print(pipeline.investigate(clean_account))

print(f"\n--- Investigating SUSPICIOUS account: {suspicious_account} ---")
print(pipeline.investigate(suspicious_account))

graph_store.close()
# Save initial adapter state so each benchmark run starts fresh.
# lora_B is zero-initialised by PEFT, so B@A=0 at the start.
# Without this reset every run after the first inherits trained weights.
import copy
_initial_adapter_state = {
    k: v.detach().clone()
    for k, v in model.named_parameters()
    if 'lora_' in k
}
print(f'Initial adapter state saved ({len(_initial_adapter_state)} tensors).')

def reset_adapter():
    """Reset LoRA weights to post-attach_lora() initial state before each run."""
    with torch.no_grad():
        for k, v in model.named_parameters():
            if k in _initial_adapter_state:
                v.copy_(_initial_adapter_state[k])
    print('Adapter reset to initial state.')


Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424
Model ready.

Building graph...
Ingesting 6924049 transactions...
  705903 unique accounts, 6924049 transactions.
Ingest complete.

--- Investigating CLEAN account: 80E121970 ---
--- Investigating account 80E121970 ---
Alright, let's dive into this transaction analysis. The user has provided a detailed log of transactions involving two accounts: 80E121970 and 80A4E09B0, as well as another account, 8008C9360. The transactions are all in US dollars, and the amounts vary, with some being labeled as "Reinvestment" and others as "Credit Card" or "Cheque".

First, I'll look at the timestamps to see if there's any unusual pattern. The transactions start on September 1st and continue through the 9th. The first few transactions are all from 80E121970 to itself, with amounts of 11.96 and 85161.18 dollars. These are labeled as "Reinvestment". Reinvestments typically involve moving funds from one account to another, oft

In [ ]:
# # The log shows samples=20 training and only 10 accounts evaluated per bank. With ~1-2% laundering rate in IBM AML, 10 eval
# #  samples gives an expected 0.1-0.2 positive examples - statistically we'll almost always see zero positives in ground truth,
# #  making F1=0 mathematically guaranteed regardless of model quality.

# # Now, serious training
# config = Config(
#     csv_path="data/LI-Small_Trans.csv",
#     graph_backend="kuzu",
#     num_clients=3,
#     num_rounds=5,
#     local_epochs=2,
#     max_train_samples=200,
#     max_eval_samples=100,
# )

# # Pass model/tokenizer from the prev. cells to avoid reloading into VRAM
# start_server(config, model=model, tokenizer=tokenizer)

Reusing provided model and tokenizer.

Ingesting 33245 transactions...
  4245 unique accounts, 33245 transactions.
Ingest complete.
[Client bank_id=1] train=806 (13 pos / 793 neg)  val=172  test=174 (3 pos)


KeyboardInterrupt: 

## Trilemma Benchmark

Full evaluation run measuring all three axes:
- **Utility** - macro F1 on the full test split per round per client
- **Efficiency** - adapter delta bytes (FLoRA) vs theoretical full-model bytes (FedAvg) per round
- **Privacy** - MIA AUC per round per client (target: near 0.5)

In [ ]:
# from src.config import Config
# from src.federation.server import start_server

# # Full benchmark config.
# # bank_ids selects the three most class-balanced partitions in the IBM AML dataset
# # (banks 20, 11, 12 - highest positive account counts at 1.4-1.9% laundering rate).
# # max_eval_samples=0 uses the full test split per client (~520/470/380 accounts each).
# # max_eval_tokens=512: enough for brief chain-of-thought + VERDICT line.
# #   - 256 is too short (reasoning gets cut off before the verdict on a fresh model)
# #   - 1024 makes eval ~2x slower with no quality benefit after fine-tuning
# # Estimated runtime on A100: ~60 min (FLoRA) + ~60 min (FedAvg) + ~50 min (centralised) = ~3 h total
# benchmark_config = Config(
#     csv_path="data/LI-Small_Trans.csv",
#     graph_backend="kuzu",
#     num_clients=3,
#     bank_ids=(20, 11, 12),  # banks with highest positive account counts
#     num_rounds=3,
#     local_epochs=1,
#     max_train_samples=100,
#     max_eval_samples=0,     # full test split - needed to capture all ~10/8/5 positives per bank
#     max_eval_tokens=1024,
#     mia_n_members=50,
#     mia_n_nonmembers=50,
# )

# history = start_server(benchmark_config, model=model, tokenizer=tokenizer)

Reusing provided model and tokenizer.

Ingesting 120114 transactions...
  14566 unique accounts, 120114 transactions.
Ingest complete.
[Client bank_id=20] train=2933 (46 pos / 2887 neg)  val=628  test=629 (10 pos)
Ingesting 123456 transactions...
  15980 unique accounts, 123456 transactions.
Ingest complete.
[Client bank_id=11] train=3387 (45 pos / 3342 neg)  val=725  test=727 (10 pos)
Ingesting 64465 transactions...
  8449 unique accounts, 64465 transactions.
Ingest complete.
[Client bank_id=12] train=1773 (25 pos / 1748 neg)  val=379  test=381 (6 pos)

Round 1/3
  [fit] bank_id=20 loss=0.1299 samples=100
  [fit] bank_id=11 loss=0.1194 samples=100
  [fit] bank_id=12 loss=0.1167 samples=100

[FLoRA] Round 1 - aggregating 3 clients
[FLoRA] Round 1 - aggregation complete

[MIA] Round 1
  [mia] bank_id=20 AUC=0.386
  [mia] bank_id=11 AUC=0.458
  [mia] bank_id=12 AUC=0.518
--- Investigating account 8014DDA20 ---
--- Investigating account 80263D820 ---
--- Investigating account 81959AB91 --

In [ ]:
from src.config import Config
from src.federation.server import start_server

# Base benchmark config (flat retrieval - raw transaction list).
# bank_ids=(20,11,12): highest positive account counts (65/56/36 pos accounts).
# max_eval_samples != 0: all positives + equal negatives (1:1 balanced).
benchmark_config = Config(
    csv_path="data/LI-Small_Trans.csv",
    graph_backend="kuzu",
    num_clients=3,
    bank_ids=(20, 11, 12),
    num_rounds=10,
    local_epochs=1,
    max_train_samples=100,
    max_eval_samples=1,
    max_eval_tokens=1024,
    mia_n_members=50,
    mia_n_nonmembers=50,
    retrieval_mode="flat",
    checkpoint_path=f"{RUN_DIR}/history_flora_flat.json",
)

history_flora_flat = start_server(benchmark_config, model=model, tokenizer=tokenizer)

import dataclasses, json
_cfg_dict = dataclasses.asdict(benchmark_config)
_cfg_dict["aggregation"] = "flora"
with open(f"{RUN_DIR}/config_flora_flat.json", "w") as f:
    json.dump(_cfg_dict, f, indent=2)
print(f"Config saved -> {RUN_DIR}/config_flora_flat.json")

model.save_pretrained(f"{RUN_DIR}/adapters/flora_flat")
print(f"Adapter saved -> {RUN_DIR}/adapters/flora_flat")


Reusing provided model and tokenizer.

Ingesting 120114 transactions...
  14566 unique accounts, 120114 transactions.
Ingest complete.
[Client bank_id=20] train=2933 (46 pos / 2887 neg)  val=628  test=629 (10 pos)
Ingesting 123456 transactions...
  15980 unique accounts, 123456 transactions.
Ingest complete.
[Client bank_id=11] train=3387 (45 pos / 3342 neg)  val=725  test=727 (10 pos)
Ingesting 64465 transactions...
  8449 unique accounts, 64465 transactions.
Ingest complete.
[Client bank_id=12] train=1773 (25 pos / 1748 neg)  val=379  test=381 (6 pos)

Round 1/10
  [fit] bank_id=20 loss=0.2062 samples=100
  [fit] bank_id=11 loss=0.1880 samples=100
  [fit] bank_id=12 loss=0.2158 samples=100

[FLoRA] Round 1 - aggregating 3 clients
[FLoRA] Round 1 - aggregation complete

[MIA] Round 1
  [mia] bank_id=20 AUC=0.436
  [mia] bank_id=11 AUC=0.438
  [mia] bank_id=12 AUC=0.472
--- Investigating account 8001B9BD0 ---
--- Investigating account 8010201B0 ---
--- Investigating account 802D2B260 -

In [ ]:
reset_adapter()

from src.federation.baselines import run_centralised

# Centralised baseline - flat retrieval. Utility upper bound (no privacy).
history_centralised_flat = run_centralised(benchmark_config, model=model, tokenizer=tokenizer)

import dataclasses, json
with open(f"{RUN_DIR}/history_centralised_flat.json", "w") as f:
    json.dump(history_centralised_flat, f)
print(f"Saved -> {RUN_DIR}/history_centralised_flat.json")

_cfg_dict = dataclasses.asdict(benchmark_config)
_cfg_dict["aggregation"] = "centralised"
with open(f"{RUN_DIR}/config_centralised_flat.json", "w") as f:
    json.dump(_cfg_dict, f, indent=2)
print(f"Config saved -> {RUN_DIR}/config_centralised_flat.json")

model.save_pretrained(f"{RUN_DIR}/adapters/centralised_flat")
print(f"Adapter saved -> {RUN_DIR}/adapters/centralised_flat")


In [ ]:
# Centralised flat history is saved inside cell 12 above (to RUN_DIR).

In [ ]:
reset_adapter()

import dataclasses, json
from src.federation.server import start_server

fedavg_flat_config = dataclasses.replace(
    benchmark_config,
    checkpoint_path=f"{RUN_DIR}/history_fedavg_flat.json",
)
history_fedavg_flat = start_server(fedavg_flat_config, model=model, tokenizer=tokenizer,
    aggregation="fedavg")

_cfg_dict = dataclasses.asdict(fedavg_flat_config)
_cfg_dict["aggregation"] = "fedavg"
with open(f"{RUN_DIR}/config_fedavg_flat.json", "w") as f:
    json.dump(_cfg_dict, f, indent=2)
print(f"Config saved -> {RUN_DIR}/config_fedavg_flat.json")

model.save_pretrained(f"{RUN_DIR}/adapters/fedavg_flat")
print(f"Adapter saved -> {RUN_DIR}/adapters/fedavg_flat")


In [ ]:
reset_adapter()

import dataclasses, json
from src.federation.server import start_server

flora_graph_config = dataclasses.replace(
    benchmark_config,
    retrieval_mode="graph",
    checkpoint_path=f"{RUN_DIR}/history_flora_graph.json",
)
history_flora_graph = start_server(flora_graph_config, model=model, tokenizer=tokenizer)

_cfg_dict = dataclasses.asdict(flora_graph_config)
_cfg_dict["aggregation"] = "flora"
with open(f"{RUN_DIR}/config_flora_graph.json", "w") as f:
    json.dump(_cfg_dict, f, indent=2)
print(f"Config saved -> {RUN_DIR}/config_flora_graph.json")

model.save_pretrained(f"{RUN_DIR}/adapters/flora_graph")
print(f"Adapter saved -> {RUN_DIR}/adapters/flora_graph")


In [ ]:
reset_adapter()

import dataclasses, json
from src.federation.baselines import run_centralised

centralised_graph_config = dataclasses.replace(
    benchmark_config,
    retrieval_mode="graph",
)
history_centralised_graph = run_centralised(centralised_graph_config, model=model, tokenizer=tokenizer)

with open(f"{RUN_DIR}/history_centralised_graph.json", "w") as f:
    json.dump(history_centralised_graph, f)
print(f"Saved -> {RUN_DIR}/history_centralised_graph.json")

_cfg_dict = dataclasses.asdict(centralised_graph_config)
_cfg_dict["aggregation"] = "centralised"
with open(f"{RUN_DIR}/config_centralised_graph.json", "w") as f:
    json.dump(_cfg_dict, f, indent=2)
print(f"Config saved -> {RUN_DIR}/config_centralised_graph.json")

model.save_pretrained(f"{RUN_DIR}/adapters/centralised_graph")
print(f"Adapter saved -> {RUN_DIR}/adapters/centralised_graph")


In [ ]:
# Centralised graph history is saved inside cell 16 above (to RUN_DIR).

In [ ]:
reset_adapter()

import dataclasses, json
from src.federation.server import start_server

fedavg_graph_config = dataclasses.replace(
    benchmark_config,
    retrieval_mode="graph",
    checkpoint_path=f"{RUN_DIR}/history_fedavg_graph.json",
)
history_fedavg_graph = start_server(fedavg_graph_config, model=model, tokenizer=tokenizer,
    aggregation="fedavg")

_cfg_dict = dataclasses.asdict(fedavg_graph_config)
_cfg_dict["aggregation"] = "fedavg"
with open(f"{RUN_DIR}/config_fedavg_graph.json", "w") as f:
    json.dump(_cfg_dict, f, indent=2)
print(f"Config saved -> {RUN_DIR}/config_fedavg_graph.json")

model.save_pretrained(f"{RUN_DIR}/adapters/fedavg_graph")
print(f"Adapter saved -> {RUN_DIR}/adapters/fedavg_graph")


In [ ]:
import json
import os

# List of expected history files
expected_files = [
    "history_flora_flat.json",
    "history_fedavg_flat.json",
    "history_centralised_flat.json",
    "history_flora_graph.json",
    "history_fedavg_graph.json",
    "history_centralised_graph.json"
]

print(f"Verifying files in: {DRIVE_DIR}\n")

for filename in expected_files:
    path = os.path.join(DRIVE_DIR, filename)
    if os.path.exists(path):
        try:
            with open(path, 'r') as f:
                data = json.load(f)
            # Determine number of rounds (usually length of 'f1' or 'train_loss' list)
            rounds = len(data.get('f1', data.get('train_loss', [])))
            print(f"[FOUND] {filename:<30} | Rounds: {rounds}")
        except Exception as e:
            print(f"[ERROR] {filename:<30} | Could not read file: {e}")
    else:
        print(f"[MISSING] {filename:<30} | Not found on Drive")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def avg_f1(h):   return [np.mean(h["f1"][r])         for r in range(len(h["f1"]))]
def avg_mia(h):  return [np.mean(h["mia_auc"][r])    for r in range(len(h["mia_auc"]))]
def avg_loss(h): return [np.mean(h["train_loss"][r])  for r in range(len(h["train_loss"]))]

rounds   = list(range(1, len(history_flora_flat["f1"]) + 1))
bank_ids = benchmark_config.bank_ids

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Trilemma Benchmark - Flat vs Graph Retrieval", fontsize=14, fontweight="bold")

# Utility
ax = axes[0, 0]
ax.plot(rounds, avg_f1(history_flora_flat),        "o-",  color="steelblue",  label="FLoRA (flat)")
ax.plot(rounds, avg_f1(history_flora_graph),       "o--", color="steelblue",  label="FLoRA (graph)")
ax.plot(rounds, avg_f1(history_fedavg_flat),       "s-",  color="darkorange", label="FedAvg (flat)")
ax.plot(rounds, avg_f1(history_fedavg_graph),      "s--", color="darkorange", label="FedAvg (graph)")
ax.plot(rounds, avg_f1(history_centralised_flat),  "^-",  color="black",      label="Centralised (flat)")
ax.plot(rounds, avg_f1(history_centralised_graph), "^--", color="black",      label="Centralised (graph)")
ax.set_title("Utility - F1 per Round (avg across banks)")
ax.set_xlabel("Round"); ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Efficiency (retrieval mode does not affect comms - adapter deltas are identical)
ax = axes[0, 1]
flora_mb  = [sum(history_flora_flat["comm_bytes_flora"][r]) / 1e6 for r in range(len(rounds))]
fedavg_mb = [sum(history_fedavg_flat["comm_bytes_flora"][r]) / 1e6 for r in range(len(rounds))]
full_mb   = history_flora_flat["comm_bytes_fedavg_per_round"] / 1e6
ax.bar([r - 0.2 for r in rounds], flora_mb,  width=0.4, label="FLoRA adapter deltas",  color="steelblue")
ax.bar([r + 0.2 for r in rounds], fedavg_mb, width=0.4, label="FedAvg adapter deltas", color="darkorange")
ax.axhline(full_mb, color="crimson", linestyle="--", linewidth=1.5,
           label=f"Full-weight FedAvg ({full_mb/1000:.0f} GB/round)")
ax.set_yscale("log")
ax.set_title("Efficiency - Communication Volume per Round (log)")
ax.set_xlabel("Round"); ax.set_ylabel("MB (log scale)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both", axis="y")

# Privacy
ax = axes[1, 0]
ax.plot(rounds, avg_mia(history_flora_flat),   "o-",  color="steelblue",  label="FLoRA (flat)")
ax.plot(rounds, avg_mia(history_flora_graph),  "o--", color="steelblue",  label="FLoRA (graph)")
ax.plot(rounds, avg_mia(history_fedavg_flat),  "s-",  color="darkorange", label="FedAvg (flat)")
ax.plot(rounds, avg_mia(history_fedavg_graph), "s--", color="darkorange", label="FedAvg (graph)")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Random (0.5 = private)")
ax.set_title("Privacy - MIA AUC per Round (avg across banks)")
ax.set_xlabel("Round"); ax.set_ylabel("AUC (closer to 0.5 = more private)")
ax.set_ylim(0.3, 1.0); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 1]
ax.plot(rounds, avg_loss(history_flora_flat),        "o-",  color="steelblue",  label="FLoRA (flat)")
ax.plot(rounds, avg_loss(history_flora_graph),       "o--", color="steelblue",  label="FLoRA (graph)")
ax.plot(rounds, avg_loss(history_fedavg_flat),       "s-",  color="darkorange", label="FedAvg (flat)")
ax.plot(rounds, avg_loss(history_fedavg_graph),      "s--", color="darkorange", label="FedAvg (graph)")
ax.plot(rounds, avg_loss(history_centralised_flat),  "^-",  color="black",      label="Centralised (flat)")
ax.plot(rounds, avg_loss(history_centralised_graph), "^--", color="black",      label="Centralised (graph)")
ax.set_title("Training Loss Convergence (avg across banks)")
ax.set_xlabel("Round"); ax.set_ylabel("Loss"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/benchmark_results.pdf", dpi=150, bbox_inches="tight")
plt.savefig(f"{DRIVE_DIR}/benchmark_results.pdf", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
def f(h):   return np.mean([h["f1"][-1][i]      for i in range(len(bank_ids))])
def mia(h): return np.mean([h["mia_auc"][-1][i] for i in range(len(bank_ids))])
def mb(h):  return sum(sum(h["comm_bytes_flora"][r]) for r in range(len(rounds))) / 1e6

full_total_mb = history_flora_flat["comm_bytes_fedavg_per_round"] * len(rounds) / 1e6
C, L = 14, 30

print()
print("=== Trilemma Summary (final round, avg across banks) ===")
print(f"{'':^{L}} {'FLoRA(flat)':^{C}} {'FedAvg(flat)':^{C}} {'Centr.(flat)':^{C}}"
      f" {'FLoRA(graph)':^{C}} {'FedAvg(graph)':^{C}} {'Centr.(graph)':^{C}}")
print("-" * (L + 6 * (C + 1)))
print(f"{'Utility (F1)':<{L}}"
      f" {f(history_flora_flat):^{C}.3f} {f(history_fedavg_flat):^{C}.3f}"
      f" {history_centralised_flat['f1'][-1][0]:^{C}.3f}"
      f" {f(history_flora_graph):^{C}.3f} {f(history_fedavg_graph):^{C}.3f}"
      f" {history_centralised_graph['f1'][-1][0]:^{C}.3f}")
print(f"{'Adapter delta MB (total)':<{L}}"
      f" {mb(history_flora_flat):^{C}.1f} {mb(history_fedavg_flat):^{C}.1f} {'N/A':^{C}}"
      f" {mb(history_flora_graph):^{C}.1f} {mb(history_fedavg_graph):^{C}.1f} {'N/A':^{C}}")
print(f"{'Param. reduction vs full-weight':<{L}}"
      f" {full_total_mb/mb(history_flora_flat):^{C}.0f}x {'1x':^{C}} {'N/A':^{C}}"
      f" {full_total_mb/mb(history_flora_graph):^{C}.0f}x {'1x':^{C}} {'N/A':^{C}}")
print(f"{'Privacy (MIA AUC)':<{L}}"
      f" {mia(history_flora_flat):^{C}.3f} {mia(history_fedavg_flat):^{C}.3f} {'N/A (no FL)':^{C}}"
      f" {mia(history_flora_graph):^{C}.3f} {mia(history_fedavg_graph):^{C}.3f} {'N/A (no FL)':^{C}}")

In [ ]:
import json, os

os.makedirs("results", exist_ok=True)

all_results = {
    "history_flora_flat.json":        globals().get("history_flora_flat"),
    "history_fedavg_flat.json":       globals().get("history_fedavg_flat"),
    "history_centralised_flat.json":  globals().get("history_centralised_flat"),
    "history_flora_graph.json":       globals().get("history_flora_graph"),
    "history_fedavg_graph.json":      globals().get("history_fedavg_graph"),
    "history_centralised_graph.json": globals().get("history_centralised_graph"),
}

for filename, data in all_results.items():
    if data is None:
        print(f"[SKIP] {filename} - not found")
        continue
    for dest in [f"results/{filename}", f"{DRIVE_DIR}/{filename}"]:
        with open(dest, "w") as f:
            json.dump(data, f)
    n = len(data.get("f1", data.get("train_loss", [])))
    print(f"[SAVED] {filename} ({n} rounds) -> Drive + local")

In [ ]:
import json, os

print(f"Verifying results on Drive at {DRIVE_DIR}...")
print()

checks = [
    ("history_flora_flat.json",        "FLoRA (flat)"),
    ("history_fedavg_flat.json",       "FedAvg (flat)"),
    ("history_centralised_flat.json",  "Centralised (flat)"),
    ("history_flora_graph.json",       "FLoRA (graph)"),
    ("history_fedavg_graph.json",      "FedAvg (graph)"),
    ("history_centralised_graph.json", "Centralised (graph)"),
]

all_ok = True
for filename, label in checks:
    fpath = f"{DRIVE_DIR}/{filename}"
    if os.path.exists(fpath):
        with open(fpath) as f:
            data = json.load(f)
        n = len(data.get("f1", data.get("train_loss", [])))
        print(f"  [OK] {label}: {n} rounds on Drive")
    else:
        print(f"  [MISSING] {label}: not found")
        all_ok = False

print()
if all_ok:
    print("All 6 results verified. Disconnecting runtime...")
    from google.colab import runtime
    runtime.unassign()
else:
    print("Some results missing - run save cell above first.")